In [1]:
import pandas as pd
import requests
from datetime import datetime, UTC
from urllib.parse import quote

In [2]:
INPUT_FILE = "rocket.csv"

In [3]:
def unix_to_date(ts):
    try:
        if ts in [None, "", 0, "0"]:
            return "NA"
        return datetime.fromtimestamp(int(ts), UTC).strftime("%Y-%m-%d %H:%M:%S")
    except:
        return "NA"

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
}

# load csv
df = pd.read_csv(INPUT_FILE, dtype=str)

# remove duplicates
df = df.drop_duplicates(subset=["host", "username", "password"])

rows = []

for _, row in df.iterrows():

    host = str(row["host"]).rstrip("/")
    username = row["username"]
    password = row["password"]

    try:
        api_url = f"{host}/player_api.php?username={quote(username)}&password={quote(password)}"

        r = requests.get(api_url, headers=headers, timeout=3)
        data = r.json()

        user_info = data.get("user_info", {})
        server_info = data.get("server_info", {})

        status = user_info.get("status")

        # skip disabled accounts
        if status and status.lower() == "disabled":
            continue

        created_at = unix_to_date(user_info.get("created_at"))
        exp_date = unix_to_date(user_info.get("exp_date"))
        active_cons = user_info.get("active_cons")
        max_connections = user_info.get("max_connections")
        url = server_info.get("url")
        server_protocol = server_info.get("server_protocol")

        row["status"] = status or "NA"
        row["created_at"] = created_at
        row["exp_date"] = exp_date
        row["active_cons"] = active_cons or "NA"
        row["max_connections"] = max_connections or "NA"
        row["url"] = url or "NA"
        row["server_protocol"] = server_protocol or "NA"

        rows.append(row)

    except Exception:
        continue

In [5]:
out_df = pd.DataFrame(rows)

# convert for sorting
out_df["active_cons"]     = pd.to_numeric(out_df["active_cons"], errors="coerce")
out_df["max_connections"] = pd.to_numeric(out_df["max_connections"], errors="coerce")
out_df["created_at"]      = pd.to_datetime(out_df["created_at"], errors="coerce")

# sort
out_df = out_df.sort_values(
    by=["host", "active_cons", "max_connections", "created_at"],
    ascending=[True, True, False, True]
)

# convert back
out_df["created_at"] = out_df["created_at"].astype(str)

# overwrite file
out_df.to_csv(INPUT_FILE, index=False)